<img src="logo.png" alt="Vegeta" width="240">

# Project Sikarian — Sikarian Lobster Ornatus, the big underwater walker

**Ornatus** (*Panulirus ornatus*, the ornate spiny lobster) is v2 of the Sikarian Lobster: the same machine as Nefri
(notebook 24) — six legs on the bottom, two pincers, one thruster in a shroud on a jointed tail — **slightly bigger**:
**15 kg**, a 0.41 m shell, a Ø100 mm propeller, a **payload bay** (a sample carousel or a sensor head: the inspection
and sampling uses noted at the end of notebook 24) and a 414 Wh pack. Its job (inputs, from the brief): **cut a
Ø12 mm electric cable** — a PVC power cable, 3×2.5 mm² copper (~2.5 kN for a blade against an anvil, the ratchet
cable-cutter class) — strung across the mouth of the **same Ø600 mm outfall pipe**, and **walk in**. The pipe is
the constraint that keeps Ornatus from being a straight 1.3× Nefri: its feet have to stay inside the pipe's 0.52 m
silt floor and its masts under the roof, so the legs are **narrower** than a scale-up.

Everything below is the same code as Nefri's with Ornatus's parameter set (`lobster_robot.ORNATUS`, a `Variant`:
design parameters, parts, servos, thruster, trim, mission tuning); the comparison with Nefri runs through the
notebook as a two-column table. Sections as notebook 24:

```
1 the scale-up: CAD side by side, both budgets, the foam and the ballast that make exactly 15.0 kg, the trim
2 in the pipe: the foot span against the silt floor, the headroom, leg loads (standing, pressing, in air), the walk
3 the thruster: Ø100 propeller in Boreas, the duct, the 180 kV motor on 4S, cavitation, the swim speed
4 the tail: the thrust-line envelope and the joint torques (why the tail servos are 8 N·m)
5 CFD (Aeromant, fast presets): the propeller at bollard vs BEMT; the body with the jet deflected (a calibrated disk)
6 the cable: the pincer's cutting envelope (the Onager Manus's chart) against cable classes, the jaw screw, jaw FEA
7 FEA: housing Ø140 at 8 m (+ buckling by hand), a leg, a tail segment; the tail's modes vs the blade pass
8 endurance: standby, watch, walk, swim, the payload
9 MuJoCo (ChironLab, water): swim and land; cut the cable and walk into the pipe; hold in a current — movies
```

*Long cells: the CFD (§5, minutes each on the fast preset; `VEGETA_SKIP_OPENFOAM=1` skips them) and the three
missions with their movies (§9, ~10 min). Everything else runs in a minute or two.*

In [ ]:
import json, math, os, shutil, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Video
from vegeta import dedalus, talos, chiron, aeromant, boreas
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz
from vegeta.chiron import viz as cviz
from vegeta.aeromant import viz as aviz
from vegeta.aeromant import movie as amovie
sys.path.insert(0, "designs")
import actuators as act
import lobster, lobster_robot as lr, lobster_controller as lc, lobster_scenario as ls, lobster_cfd as cfd

RUNS = Path("_runs/sikarian_lobster_ornatus"); shutil.rmtree(RUNS, ignore_errors=True); RUNS.mkdir(parents=True)
OUT = Path("output/25_sikarian_lobster_ornatus"); OUT.mkdir(parents=True, exist_ok=True)
RUN_CFD = os.environ.get("VEGETA_SKIP_OPENFOAM", "0") != "1"
G, RHO = lr.WATER["g"], lr.WATER["density"]
design = lobster.SikarianLobster()
V, N = lr.ORNATUS, lr.NEFRI_V                      # the two members
p, pn = dict(V.params), dict(N.params)
MS = V.mission
JOB = pd.Series({"name": V.display + " (v2, big)", "water": "fresh, 0-8 m, 1000 kg/m³", "mass [kg]": V.target_mass,
                 "the cable": f"{ls.CABLE['name']}, cut at {ls.CABLE['f_cut_N']:.0f} N", "the pipe": f"Ø{ls.PIPE['inner_d'] * 1000:.0f} mm inner, {ls.PIPE['silt'] * 1000:.0f} mm of silt in it",
                 "current to hold in [m/s]": 0.5, "payload bay [kg]": V.parts_kg["payload bay (sample carousel / sensor head, in the shell)"]},
                name="Ornatus — the job (inputs)")
display(JOB.to_frame())
both = pd.DataFrame({"Nefri": pn, "Ornatus": p}).loc[[k for k in lr.NEFRI]]
both["Ornatus / Nefri"] = (both["Ornatus"] / both["Nefri"]).replace([np.inf, -np.inf], np.nan)
both[both["Ornatus"] != both["Nefri"]].round(3)

## 1. The scale-up: CAD, budgets, foam and ballast, trim

Nefri's shell, housing, legs, arms, jaws and tail scaled by ~1.28 — except the legs (femur 45 instead of a scaled 90,
tibia splay 4° instead of 10°: §2 says why) — with a Ø140 × 280 mm pressure housing (3 mm wall), a 1 kg payload
bay in the shell aft of the housing, the 4S6P pack (1.6 kg), a bigger sonar, a Ø100 mm propeller in a Ø106 shroud,
sealed 3 N·m leg servos, 8 N·m arm and tail servos, and a **sealed 4 kN lead-screw jaw drive** on each pincer (the
Onager Manus's ball screw at a tenth of the force; self-locking, so a grip holds unpowered).

**15.0 kg exactly.** The budget solves two things at once: the syntactic foam that makes the robot sink with
`NET_FRACTION` = 10 % of its weight, and steel **ballast plates** that bring the total to the target mass
(`lr.budget(variant="ornatus")`: ``ballast_kg``). Little ballast means the design is close to its mass budget;
``over_target_kg`` > 0 would mean it is over. **Trim** as Nefri: the foam high and forward (its x solved so the centre
of buoyancy sits over the centre of gravity standing), the battery and the payload low and aft — the jaw screws make
the pincers heavy forward.

In [ ]:
parts = {k: design.generate(**dict(p, part=k)) for k in ("robot", "shell", "housing", "leg", "upper_arm", "jaw", "tail_segment", "shroud", "propeller")}
cad = {k: g.export(RUNS / "cad" / k, stl_tolerance=0.1 if k in ("jaw", "propeller") else 0.3) for k, g in parts.items()}
cad_num = lr.cad_numbers(p, recompute=True)
assert max(abs(cad_num[k] / V.cad[k] - 1) for k in V.cad) < 0.01, "lobster_robot.ORNATUS_CAD is stale: update it from cad_num"
nefri_cad = design.generate(**dict(pn, part="robot"))
dviz.show(dviz.plot3d(parts["robot"]))

In [ ]:
b = lr.budget(p, cad_num, V)
bn = lr.budget(variant=N)
mass = pd.DataFrame({"mass [kg]": {k: v[0] for k, v in b["parts"].items()}, "volume [L]": {k: v[1] * 1000 for k, v in b["parts"].items()}})
mass["buoyancy − weight [N]"] = (RHO * mass["volume [L]"] / 1000 - mass["mass [kg]"]) * G
mass.loc["TOTAL"] = mass.sum()
robot = lr.lobster(variant=V, cad=cad_num)
robot_n = lr.lobster(variant=N)
compare = pd.DataFrame({"Nefri": {"mass [kg]": bn["mass"], "displaced [L]": bn["volume"] * 1000, "wet weight [N]": bn["wet_weight_N"], "foam [L]": bn["foam_volume"] * 1000,
                                  "foam room [L]": bn["foam_space"] * 1000, "ballast [kg]": bn["ballast_kg"], "foam centre x [mm]": robot_n.trim["foam_x_m"] * 1000,
                                  "servos [kg]": sum(v[0] for k, v in bn["parts"].items() if "servos" in k), "battery [Wh]": N.thruster["battery"].energy_wh},
                        "Ornatus": {"mass [kg]": b["mass"], "displaced [L]": b["volume"] * 1000, "wet weight [N]": b["wet_weight_N"], "foam [L]": b["foam_volume"] * 1000,
                                    "foam room [L]": b["foam_space"] * 1000, "ballast [kg]": b["ballast_kg"], "foam centre x [mm]": robot.trim["foam_x_m"] * 1000,
                                    "servos [kg]": sum(v[0] for k, v in b["parts"].items() if "servos" in k), "battery [Wh]": V.thruster["battery"].energy_wh}})
print(f"Ornatus {b['mass']:.2f} kg with {b['ballast_kg'] * 1000:.0f} g of ballast (over target: {b['over_target_kg']:.2f} kg) | foam {b['foam_volume'] * 1000:.2f} L in "
      f"{b['foam_space'] * 1000:.2f} L of room, its centre at x = {robot.trim['foam_x_m'] * 1000:+.0f} mm (the shell ends at ±{p['shell_length'] / 2:.0f}) | "
      f"the pitching couple {robot.trim['couple_untrimmed_Nm']:+.2f} N·m untrimmed")
display(compare.round(2))
mass.round(3)

In [ ]:
lab_flat = lr.lobster_lab(chiron.Flat(), robot=robot)
ep_stand = lab_flat.run(lc.Mission([lc.Phase("stand", lc.wait(3.0), None, 4.0)], name="stand"), duration=3.0, rules=None, settle=0.0)
fz = np.asarray(ep_stand.log["foot_force"])[-1, :, 2]
bq = np.asarray(ep_stand.log["body_quat"])[-1, 0]
names = list(ep_stand.log["joints"]); tau0 = np.asarray(ep_stand.log["tau"])[-1]
ARM = act.get(V.servo_keys["arm"])
print(f"standing in water (MuJoCo, 3 s): feet carry {fz.sum():.2f} N (the wet weight {b['wet_weight_N']:.2f} N), per leg {fz.round(2)} N; "
      f"pitch {math.degrees(lc.pitch_of(bq)):+.2f}°, hull centre {np.asarray(ep_stand.log['body_pos'])[-1, 0, 2] * 1000:.0f} mm over the bed | "
      f"the stowed arm's shoulder holds {abs(tau0[names.index('R_shoulder')]):.2f} N·m ({ARM.key}: rated {ARM.rated_Nm}, stall {ARM.stall_Nm})")

## 2. In the pipe: feet, headroom, leg loads, the walk

The pipe is Ø600 inner with 150 mm of silt: its **floor** is 0.52 m wide and the roof is 0.45 m over the silt on
the centre line, less to the sides. A 1.28× Nefri would stand 0.56 m wide at the feet; Ornatus's hips are 34 mm
further out than Nefri's but its femur is shorter and its tibia nearly vertical: **0.45 m** over the feet — and
what counts is the outer edge of a foot against the lowest staves' inner corners: 0.22 m, Nefri's margin (a first
try at femur 55 scraped the staves at the mouth and stalled), the mast tops at 0.39 m (5 cm under the roof there).

**Leg loads** as in notebook 24: on a tripod each foot carries a third of the wet weight — or of the wet weight plus
the thrust when it presses — at the hip's reach; in air on six legs or a tripod. Ornatus keeps the 3 N·m leg
servos (the 8 N·m class would cost 1 kg of its 15): in water it has margin; **in air it lies on its belly** (both do).
**The walk**: the tripod gait at Ornatus's `v_nominal` 0.12 m/s and a 1.3 s cycle.

In [ ]:
L_ = lobster.SikarianLobster
r_in, silt = ls.PIPE["inner_d"] / 2, ls.PIPE["silt"]
floor_w = 2 * math.sqrt(r_in ** 2 - (r_in - silt) ** 2)
rows = {}
for name, pp in (("Nefri", pn), ("Ornatus", p), ("Nefri × 1.28 (a pure scale-up)", {k: (v * 1.28 if isinstance(v, float) and k not in ("prop_blades", "tibia_splay_deg", "tail_yaw_deg", "tail_pitch_deg", "claw_pitch_deg", "claw_elbow_deg", "claw_wrist_deg", "jaw_open_deg") else v) for k, v in pn.items()})):
    fo = L_.foot_offset(pp)
    span = 2 * (L_.hips(pp)["FL"][1] + fo[1]) / 1000 + pp["foot_diameter"] / 1000
    stand = ls._standing_height(pp)
    y_mast = pp["shell_width"] * 0.2 / 1000
    roof = (r_in - silt) + math.sqrt(r_in ** 2 - y_mast ** 2)
    mast_top = stand + pp["shell_height"] / 2000 + pp["mast_height"] / 1000
    rows[name] = {"foot span [m]": span, "silt floor [m]": floor_w, "fits the floor": span < floor_w - 0.04, "hull centre standing [m]": stand,
                  "mast top [m]": mast_top, "roof at the masts [m]": roof, "fits the roof": mast_top < roof - 0.03}
pipe_fit = pd.DataFrame(rows).T
display(pipe_fit.round(3))
g = lr.geometry(p); gn = lr.geometry(pn)
tab = lr.thruster_table(p, rpm=np.linspace(0, V.thruster["rpm_max"], 23), speed=np.array([0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5]), variant=V)
tab_n = lr.thruster_table(pn, variant=N)
T_press, T_press_n = lr.thrust(tab, MS["press_rpm"], 0.0)[0], lr.thrust(tab_n, N.mission["press_rpm"], 0.0)[0]
LEG, LEG_N = act.get(V.servo_keys["leg"]), act.get(N.servo_keys["leg"])
legs = {}
for name, bb, gg, Tp, srv in (("Nefri", bn, gn, T_press_n, LEG_N), ("Ornatus", b, g, T_press, LEG)):
    W_wet, W_air, reach = bb["wet_weight_N"], bb["mass"] * G, gg["foot"][1]
    for case, F in (("standing in water, tripod", W_wet / 3), (f"pressing ({Tp:.0f} N of thrust), tripod", (W_wet + Tp) / 3),
                    ("in air on six legs", W_air / 6), ("in air on a tripod", W_air / 3)):
        legs[(name, case)] = {"foot load [N]": F, "hip lift [N·m]": F * reach, "stall / torque": srv.stall_Nm / (F * reach), "rated / torque": srv.rated_Nm / (F * reach), "servo": srv.key}
legs = pd.DataFrame(legs).T
legs.round(2)

In [ ]:
lab_walk = lr.lobster_lab(chiron.Flat(), robot=robot)
walk = lc.Mission([lc.Phase("stand", lc.wait(0.5), None, 1.0), lc.Phase("walk", lc.walk_to(0.8, v=MS["walk_v"]), None, 15.0),
                   lc.Phase("stand", lc.wait(0.6), None, 1.0)], name="walk 0.8 m")
ep_walk = lab_walk.run(walk, duration=11.0, rules=None, settle=0.0)
tw = ls.timeseries(ep_walk)
st = {r[1]: r[0] for r in walk.log if r[2] == "start"}; dn = [r[0] for r in walk.log if r[1] == "walk" and r[2] == "done"]
t0, t1 = st["walk"], (dn[0] if dn else tw.t.iloc[-1])
sel = tw[(tw.t > t0 + 1.3) & (tw.t < t1)]
v_walk = (sel.x.iloc[-1] - sel.x.iloc[0]) / (sel.t.iloc[-1] - sel.t.iloc[0])
names = list(ep_walk.log["joints"]); tau = np.abs(np.asarray(ep_walk.log["tau"]))
lift_peak = max(tau[:, names.index(f"{leg}_lift")].max() for leg in lr.LEGS)
fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
ax[0].plot(tw.t, tw.x, label="hull x [m]"); ax[0].plot(tw.t, tw.tilt_deg / 10, label="tilt / 10 [deg]"); ax[0].plot(tw.t, tw.foot_load_N / 10, label="feet / 10 [N]")
ax[0].set(xlabel="time [s]", title=f"tripod walk: {v_walk:.3f} m/s (commanded {MS['walk_v']})"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
for leg in lr.LEGS:
    ax[1].plot(ep_walk.log["t"], tau[:, names.index(f"{leg}_lift")], lw=0.8, label=leg)
ax[1].axhline(LEG.rated_Nm, color="k", ls="--", lw=0.8); ax[1].set(xlabel="time [s]", ylabel="|τ| hip lift [N·m]", title="hip lift servos (dashed: rated)"); ax[1].legend(fontsize=7, ncol=3); ax[1].grid(alpha=0.3)
fig.tight_layout()
print(f"walking {v_walk:.3f} m/s (Nefri 0.10); peak hip lift torque {lift_peak:.2f} N·m (rated {LEG.rated_Nm}, stall {LEG.stall_Nm})")

## 3. The thruster: Ø100 propeller in water, the duct, cavitation, the swim

Boreas blade element theory (the marine section of notebooks 13/14) on the Ø100 mm, 92 mm pitch, three-blade
propeller at 0–2800 rpm: thrust, torque, the 180 kV motor's current and efficiency on the 4S pack — 25 A at full
speed, so the thruster is run at 2000 rpm for the swim and 1800 for the press. The duct's gain stays 1.2 (notebook
24 §3). Cavitation at 0.7 R against −C_p,min = 1 at the surface and at 8 m; the swim speed from thrust = drag with the
body's C_d·A scaled from Nefri's by the frontal area (an input).

In [ ]:
mot, bat = V.thruster["motor"], V.thruster["battery"]
FW = boreas.noise.Medium("fresh water", RHO, 1480.0)
prop = lr._prop(p)
fig, ax = plt.subplots(1, 3, figsize=(17, 4.2))
for j, v in enumerate(tab["speed"]):
    ax[0].plot(tab["rpm"], tab["thrust"][:, j] * lr.DUCT_GAIN, label=f"{v:.2f} m/s")
ax[0].plot(tab_n["rpm"], tab_n["thrust"][:, 0] * lr.DUCT_GAIN, "k:", label="Nefri, bollard")
ax[0].set(xlabel="rpm", ylabel="thrust (ducted) [N]", title="thrust × DUCT_GAIN vs rpm and inflow"); ax[0].legend(fontsize=7); ax[0].grid(alpha=0.3)
rows = {}
for rpm in (1200, 1500, 1800, 2000, 2400, 2800):
    T, Q = lr.thrust(tab, rpm, 0.0)
    Vm, I = mot.voltage_for(rpm, Q)
    rows[rpm] = {"thrust (ducted) [N]": T, "torque [N·m]": Q, "shaft [W]": Q * rpm * 2 * math.pi / 60, "motor [V]": Vm, "motor [A]": I,
                 "electric [W]": Vm * I, "motor η": mot.efficiency(rpm, Q), "fits the pack": Vm <= bat.voltage * 1.02 and I <= mot.max_current_a}
bollard = pd.DataFrame(rows).T
depths = np.linspace(0, 8, 33)
for rpm in (2000, 2800):
    ax[1].plot(depths, [boreas.cavitation(prop, rpm, 0.0, d, FW)["cavitation_number"] for d in depths], label=f"{rpm} rpm")
ax[1].axhline(1.0, color="r", ls="--", lw=0.8, label="−Cp,min = 1: cavitation below")
ax[1].set(xlabel="depth [m]", ylabel="σ at 0.7 R", title="cavitation number at bollard"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
CDA_N = 0.069                                                                          # Nefri's, notebook 24 §3 (from MuJoCo's fluid model)
CDA = CDA_N * (p["shell_width"] * p["shell_height"]) / (pn["shell_width"] * pn["shell_height"])
vs = np.linspace(0, 1.5, 61)
for rpm in (1500, 2000, 2800):
    ax[2].plot(vs, [lr.thrust(tab, rpm, v)[0] for v in vs], label=f"thrust {rpm} rpm")
ax[2].plot(vs, 0.5 * RHO * CDA * vs ** 2, "k", label=f"drag (C_d·A {CDA:.3f} m²)")
ax[2].set(xlabel="speed [m/s]", ylabel="N", title="the swim: thrust = drag", ylim=(0, 90)); ax[2].legend(fontsize=8); ax[2].grid(alpha=0.3)
fig.tight_layout()
v_swim = {rpm: round(float(vs[np.argmin(np.abs([lr.thrust(tab, rpm, v)[0] - 0.5 * RHO * CDA * v ** 2 for v in vs]))]), 2) for rpm in (1500, 2000, 2800)}
cav = pd.DataFrame({f"{d:g} m": {rpm: boreas.cavitation(prop, rpm, 0.0, d, FW)["cavitation_number"] for rpm in (1800, 2000, 2800)} for d in (0, 1, 8)})
print(f"swim speed {v_swim} m/s (Nefri 0.8 at 2600) | blade pass at 2800 rpm {3 * 2800 / 60:.0f} Hz | pack {bat.energy_wh:.0f} Wh")
display(cav.round(2).rename_axis("σ at rpm"))
bollard.round(3)

## 4. The tail: the thrust-line envelope and the joint torques

The same geometry as Nefri's at 1.28× (tail joints at the same places relative to the shell), so the same envelope:
through the CG only within about −25°…+15°, steep downward thrust with the tail curled over the back and the line
through a point between the feet (Ornatus: `press_x` −0.10 m). What changes is the **torque**: 36 N of thrust at
the press on a 110 mm first segment is 3 N·m at joint 1 — Nefri's 3 N·m servo would be at stall; Ornatus's tail has
the **8 N·m** class. The chart shows the joint-1 moment over the envelope at the press rpm and at full speed.

In [ ]:
thetas = np.arange(-90, 91, 5)
TAIL = act.get(V.servo_keys["tail"])
env_rows = {}
for th in thetas:
    q = lc.thrust_line(math.radians(th), (0.0, 0.0, 0.0), g)
    qc = lc.thrust_line(math.radians(th), (MS["press_x"], 0.0, 0.0), g)
    row = {"through the CG": q is not None, "between the feet": qc is not None}
    for name, qq in (("CG", q), ("feet", qc)):
        if qq is not None:
            row[f"q1 {name} [deg]"], row[f"q2 {name} [deg]"] = math.degrees(qq[0]), math.degrees(qq[1])
    env_rows[th] = row
envelope = pd.DataFrame(env_rows).T
Tmax = lr.thrust(tab, V.thruster["rpm_max"], 0.0)[0]
def joint1(q2, T):
    return T * g["L1"] * abs(math.sin(q2))
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
ok = envelope[envelope["through the CG"] == True]
okc = envelope[envelope["between the feet"] == True]
ax[0].plot(ok.index, ok["q1 CG [deg]"], "o-", label="q1 (through the CG)"); ax[0].plot(ok.index, ok["q2 CG [deg]"], "o-", label="q2 (through the CG)")
ax[0].plot(okc.index, okc["q1 feet [deg]"], "s--", label="q1 (between the feet)"); ax[0].plot(okc.index, okc["q2 feet [deg]"], "s--", label="q2 (between the feet)")
ax[0].set(xlabel="thrust angle θ above the hull's x [deg]", ylabel="tail pitch [deg]", title="the tail's thrust-line envelope"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
for T, lab_ in ((T_press, f"{MS['press_rpm']:.0f} rpm (the press)"), (Tmax, f"{V.thruster['rpm_max']:.0f} rpm")):
    ax[1].plot(okc.index, [joint1(math.radians(v), T) for v in okc["q2 feet [deg]"]], label=f"{lab_}, between the feet")
ax[1].axhline(TAIL.stall_Nm, color="r", ls="--", lw=0.8, label=f"tail servo stall ({TAIL.key})"); ax[1].axhline(TAIL.rated_Nm, color="k", ls=":", lw=0.8, label="rated")
ax[1].axhline(act.get(N.servo_keys["tail"]).stall_Nm, color="#999", ls="--", lw=0.8, label="Nefri's tail servo stall")
ax[1].set(xlabel="θ [deg]", ylabel="joint 1 [N·m]", title="the thrust's moment at tail joint 1 (bollard)"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
fig.tight_layout()
q_press = lc.thrust_line(math.radians(MS["press_deg"]), (MS["press_x"], 0, 0), g)
print(f"through the CG: θ from {ok.index.min()}° to {ok.index.max()}° | between the feet: {okc.index.min()}° to {okc.index.max()}° | "
      f"the press (θ {MS['press_deg']:.0f}°, {MS['press_rpm']:.0f} rpm, {T_press:.0f} N): q = {np.degrees(q_press).round(0)}°, joint 1 {joint1(q_press[1], T_press):.2f} N·m | "
      f"propeller torque {lr.thrust(tab, MS['swim_rpm'], 0)[1]:.2f} N·m at {MS['swim_rpm']:.0f} rpm")

## 5. CFD (Aeromant, OpenFOAM): the propeller in water, and the vectored jet on the body

The two cases of notebook 24 §5 with Ornatus's geometry (`lobster_cfd` with `variant="ornatus"`): the bare Ø100
propeller at bollard in a rotating frame (`rotor_mrf_static`) against Boreas, with its particle movie; and the whole
body swimming at 0.6 m/s with the propeller as a **calibrated rotor disk** (`hull_rotor_disk`, `fixed_inflow`: the
disk runs at the rpm whose no-induction thrust is the BEMT thrust at 2000 rpm), the tail straight and yawed 35° —
the jet scrubbing the shell, the side force and the yaw moment on the body against `T sin δ`. Fast presets.

In [ ]:
env = None
try:
    env = aeromant.OpenFOAMEnvironment.detect()
    print("OpenFOAM", env.version(), env.flavor())
except Exception as exc:
    print("no OpenFOAM:", exc); RUN_CFD = False
CFD_QUALITY = "fast"
prop_stl = cfd.prop_stl(RUNS / "cad", variant=V)
case_prop = cfd.prop_case(prop_stl, RUNS / "cfd_prop", p, quality=CFD_QUALITY, environment=env, variant=V)
print(case_prop.prepare(overwrite=True))
aviz.show(aviz.plot_setup(case_prop))

In [ ]:
bemt = cfd.bemt_bollard(p=p, variant=V)
res_prop = None
if RUN_CFD:
    res_prop = case_prop.run(processors=4, progress=True)
    print(res_prop.ok, res_prop.messages[-2:])
comp = {"BEMT (Boreas), open": bemt}
if res_prop is not None and res_prop.ok:
    mp = res_prop.metrics
    comp["CFD (rotor_mrf_static), open"] = {"rpm": bemt["rpm"], "thrust_N": mp["thrust_N"], "torque_Nm": mp["torque_Nm"], "power_W": mp["power_W"]}
    print(f"{mp.get('mesh_cells')} cells, converged {mp.get('converged')} in {mp.get('iterations')} iterations; figure of merit {mp.get('figure_of_merit', float('nan')):.2f}")
compare_prop = pd.DataFrame(comp).T
if len(comp) > 1:
    compare_prop.loc["CFD / BEMT"] = compare_prop.iloc[1] / compare_prop.iloc[0]
display(compare_prop.round(3))
if res_prop is not None and res_prop.ok:
    aviz.show(aviz.plot_field_slice(case_prop, "U", normal="z"))
    clip = amovie.make_movie(case_prop, OUT / "ornatus_propeller_bollard.mp4", blades=int(p["prop_blades"]), n=40, seconds=8, fps=24, degrees_per_frame=10,
                             title=f"Ornatus's 100 mm propeller at bollard, {bemt['rpm']:.0f} rpm in water (rotor_mrf_static): tracer particles")
    display(Video(str(clip), embed=False, width=900))

In [ ]:
DEFLECTIONS = (0.0, 35.0)
tail_cases, tail_res = {}, {}
for d in DEFLECTIONS:
    stl = cfd.body_stl(RUNS / "cad", d, p, variant=V)
    tail_cases[d] = cfd.tail_case(stl, RUNS / f"cfd_tail_{d:g}", d, p, quality=CFD_QUALITY, environment=env, variant=V)
    pr = tail_cases[d].prepare(overwrite=True)
    cal = cfd.disk_calibration(d, p, variant=V)
    print(f"jet deflected {d:g}°: {pr.ok}, disk at {np.round(tail_cases[d].parameters['disk1_center'], 3).tolist()} m, axis {np.round(tail_cases[d].parameters['disk_axis'], 3).tolist()}; "
          f"BEMT {cal['target_thrust_N']:.1f} N at {cal['rpm']:.0f} rpm → the disk at {cal['disk_rpm']:.0f} rpm (no induction: {cal['uncalibrated_thrust_N']:.1f} N at {cal['rpm']:.0f})")
aviz.show(aviz.plot_setup(tail_cases[35.0]))

In [ ]:
rows = {}
if RUN_CFD:
    for d, case in tail_cases.items():
        tail_res[d] = case.run(processors=4, progress=True)
        print(d, tail_res[d].ok, tail_res[d].messages[-2:])
        if tail_res[d].ok:
            f = cfd.tail_forces(case)
            T_swim = cfd.disk_calibration(d, p, variant=V)["target_thrust_N"]
            simple = cfd.jet_side_force(d, T_swim)
            rows[f"{d:g}°"] = {"body drag [N]": f["drag_N"], "body side force, robot y [N]": f["force_robot_N"][1], "body lift [N]": f["lift_N"],
                               "body yaw moment [N·m]": f["yaw_moment_Nm"], "disk thrust (solver log) [N]": f.get("effective_lift", float("nan")),
                               "BEMT thrust, open [N]": T_swim, "T sin δ [N]": simple["side_N"], "cells": tail_res[d].metrics.get("mesh_cells")}
jet = pd.DataFrame(rows).T
display(jet.round(3))
if 35.0 in tail_res and tail_res[35.0].ok:
    aviz.show(aviz.plot_field_slice(tail_cases[35.0], "U", normal="z", origin=(0, 0, float(tail_cases[35.0].parameters["disk1_center"][2]))))
    clip = aviz.animate_particles(tail_cases[35.0], OUT / "ornatus_tail_jet_35deg.mp4", n=400, seconds=8, fps=24, lengths=4.0, progress=True)
    display(Video(str(clip), embed=False, width=900))

## 6. The cable: the cutting envelope, the jaw screw, the jaw under load

The Onager Manus's pincer check (notebook 22 §3) at Ornatus's size. The jaw drive is a **sealed 4 kN lead screw on a
20 mm lever**, given at the jaw pin as 80 N·m stall / 30 N·m rated (`sealed jaw screw 4 kN`; self-locking: it holds
a grip unpowered, and the cut does not depend on holding current). The squeeze along the jaw is `τ / x`; the
cutter notch sits 18 mm from the pin: **4.4 kN** at stall. The cable classes are inputs (blade-against-anvil cutting
forces of the class — a cutter test on the actual cable sets them): the chart is the envelope, the table says which
cables Ornatus cuts in the notch and how far out along the jaw. In MuJoCo (§9) the cable parts when both jaws press
it with 2.5 kN for 100 ms (`lobster_scenario.CABLE`, the Manus's `WireCutter` hook) — the contact forces decide; the
notch's 5 mm outer tooth keeps the round cable from being pushed out along the blade.

**Jaw FEA** (tool steel, its own frame): the pin bore held, the stall squeeze on the notch.

In [ ]:
JAW = act.get(V.servo_keys["jaw"])
F_stall, F_rated = JAW.stall_Nm / g["cutter"], JAW.rated_Nm / g["cutter"]
CABLES = {"PP rope Ø10 (Nefri's job)": 600.0, "PVC power cable 3×2.5 mm², Ø12 (the job)": ls.CABLE["f_cut_N"], "PVC power cable 3×6 mm², Ø15": 4000.0,
          "subsea signal/tether cable Ø12 (aramid member)": 3500.0, "steel-wire-armoured cable Ø12 (SWA)": 7000.0,
          "HMPE (Dyneema) line Ø8": 2500.0}                                            # inputs: cutting forces of the class [N]
xs = np.linspace(0.008, g["Lj"], 100)
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(xs * 1000, JAW.stall_Nm / xs / 1000, "k", lw=2, label="squeeze at stall, τ / x")
ax.plot(xs * 1000, JAW.rated_Nm / xs / 1000, "k:", lw=1, label="squeeze at the rated torque")
ax.plot(xs * 1000, act.get(N.servo_keys["jaw"]).stall_Nm / xs / 1000, color="#999", lw=1, label="Nefri's worm drive at stall")
cut_tab = {}
for (name, F), col in zip(CABLES.items(), ("#90caf9", "#c62828", "#ef6c00", "#6a1b9a", "#6d4c41", "#2e7d32")):
    ax.axhline(F / 1000, color=col, ls="--", lw=1, label=f"{name}: {F / 1000:.1f} kN")
    x_max = JAW.stall_Nm / F
    cut_tab[name] = {"cutting force [kN]": F / 1000, "cuts up to x [mm] from the pin": x_max * 1000, "in the notch (18 mm)": x_max >= g["cutter"],
                     "Nefri cuts it": act.get(N.servo_keys["jaw"]).stall_Nm / gn["cutter"] >= F}
ax.axvline(g["cutter"] * 1000, color="k", ls=":"); ax.text(g["cutter"] * 1000, 8, " notch", fontsize=8)
ax.set(xlabel="distance from the jaw pin [mm]", ylabel="kN", ylim=(0, 10), title="Ornatus's cutting envelope"); ax.legend(fontsize=7); ax.grid(alpha=0.3)
cables = pd.DataFrame(cut_tab).T
print(f"squeeze at the notch: {F_stall:.0f} N at stall, {F_rated:.0f} N rated ({JAW.key}; Nefri: {act.get(N.servo_keys['jaw']).stall_Nm / gn['cutter']:.0f} N)")
display(cables.round(2))
TOOL = talos.Material("tool steel jaw (42CrMo4 QT)", youngs_modulus=210000.0, poissons_ratio=0.30, density=7.85e-9, yield_strength=900.0, source="EN 10083-3")
jt, jd, nx_, nd_ = p["jaw_thickness"], p["jaw_depth"], p["cutter_x"], p["cutter_depth"]
rp = p["jaw_pin_diameter"] / 2 + 0.5
JAW_REGIONS = [talos.SurfacesInBox("pin", (-rp, -jt, jd / 2 - rp, rp, jt, jd / 2 + rp)),
               talos.SurfacesInBox("notch", (nx_ - nd_ - 0.5, -jt, -1.5, nx_ + nd_ + 0.5, jt, nd_ + 0.5))]
jm = talos.StructuralModel(cad["jaw"].artifacts["step"], "mm-N-MPa", TOOL, JAW_REGIONS, [talos.FixedSupport("pin")], [talos.Force("notch", fz=F_stall)],
                           talos.MeshSettings(element_size=2.0, min_element_size=0.5), name="jaw_cut")
jm.mesh(RUNS / "jaw_cut"); jaw_res = jm.solve(RUNS / "jaw_cut")
print(f"jaw at {F_stall:.0f} N on the notch: max von Mises {jaw_res.metrics.get('max_von_mises'):.0f} MPa, SF {jaw_res.metrics.get('safety_factor_yield'):.2f}, "
      f"max displacement {jaw_res.metrics.get('max_displacement'):.3f} mm")
tviz.show(tviz.plot_results(jaw_res, field="von_mises"))

## 7. FEA (Talos): the housing at 8 m, a leg, a tail segment; the tail's modes

As notebook 24 §7 at Ornatus's size: the Ø140 × 280 mm **housing** with a 3 mm wall under 1.5 × the pressure at 8 m
(and the hoop stress and the long-tube buckling pressure by hand — the bigger tube is nearer its buckling limit, but
still far), a **leg** (PA12-CF) under the pressing tripod's load × 2 plus the current's share, **tail segment 1** with
the full bollard thrust across its end, and the tail's modes with the rest of the tail (and the water in the shroud)
lumped at its end against the shaft and blade-pass orders.

In [ ]:
AL6082 = talos.Material("Al 6082-T6 tube", youngs_modulus=70000.0, poissons_ratio=0.33, density=2.7e-9, yield_strength=260.0, source="EN 755-2")
PA12CF = talos.Material("PA12-CF (30 % carbon, printed)", youngs_modulus=7000.0, poissons_ratio=0.35, density=1.15e-9, yield_strength=70.0, source="assumed: datasheet class, printed in-plane")
depth, SF_p = 8.0, 1.5
p_ext = RHO * G * depth * SF_p * 1e-6                                # MPa
r_o, t_w, L_h = p["housing_diameter"] / 2, p["housing_wall"], p["housing_length"]
info = talos.inspect_step(cad["housing"].artifacts["step"], "mm-N-MPa")
outer = [s.tag for s in info.surfaces if s.kind != "Plane" and s.bbox_max[1] > r_o - 0.2]
HOUSING_REGIONS = [talos.Surfaces("outer_tube", outer), talos.SurfacesOnPlane("front", "x", L_h / 2), talos.SurfacesOnPlane("rear", "x", -L_h / 2)]
hmod = talos.StructuralModel(cad["housing"].artifacts["step"], "mm-N-MPa", AL6082, HOUSING_REGIONS, [talos.FixedSupport("rear")],
                             [talos.Pressure("outer_tube", p_ext), talos.Pressure("front", p_ext)], talos.MeshSettings(element_size=5.0, min_element_size=1.5), name="housing_8m")
hmod.mesh(RUNS / "housing_8m"); housing_res = hmod.solve(RUNS / "housing_8m")
r_m = r_o - t_w / 2
sigma_hoop = p_ext * r_m / t_w
p_cr = AL6082.youngs_modulus / (4 * (1 - AL6082.poissons_ratio ** 2)) * (t_w / r_m) ** 3
W_wet = b["wet_weight_N"]
A_side = g["shell"][0] * g["shell"][2] * 1.3; F_cur = 0.5 * RHO * 1.0 * A_side * 0.5 ** 2
leg_F = (W_wet + T_press) / 3 * 2.0; leg_side = F_cur / 3
fx_, fy_, fz_ = lobster.SikarianLobster.foot_offset(p); rf = p["foot_diameter"] / 2 + 0.5; rb = p["leg_width"] * 0.8 + 0.5
LEG_REGIONS = [talos.SurfacesInBox("hip", (-rb, -rb, -rb, rb, rb, rb)), talos.SurfacesInBox("foot", (fx_ - rf, fy_ - rf, fz_ - rf, fx_ + rf, fy_ + rf, fz_ + rf))]
lmod = talos.StructuralModel(cad["leg"].artifacts["step"], "mm-N-MPa", PA12CF, LEG_REGIONS, [talos.FixedSupport("hip")],
                             [talos.Force("foot", fy=leg_side, fz=leg_F)], talos.MeshSettings(element_size=4.0, min_element_size=1.2), name="leg_press")
lmod.mesh(RUNS / "leg_press"); leg_res = lmod.solve(RUNS / "leg_press")
L1, dt_ = p["tail_segment_1"], p["tail_diameter"]
TAIL_REGIONS = [talos.SurfacesInBox("knuckle", (-dt_ * 0.45, -dt_ * 0.45, -dt_ * 0.45, dt_ * 0.45, dt_ * 0.45, dt_ * 0.45)), talos.SurfacesOnPlane("end", "x", -L1)]
tmod = talos.StructuralModel(cad["tail_segment"].artifacts["step"], "mm-N-MPa", PA12CF, TAIL_REGIONS, [talos.FixedSupport("knuckle")],
                             [talos.Force("end", fz=Tmax)], talos.MeshSettings(element_size=5.0, min_element_size=1.8), name="tail_thrust")
tmod.mesh(RUNS / "tail_thrust"); tail_res_fea = tmod.solve(RUNS / "tail_thrust")
def row(res, load):
    mt = res.metrics
    return {"load": load, "max von Mises [MPa]": mt.get("max_von_mises"), "SF yield": mt.get("safety_factor_yield"), "max displacement [mm]": mt.get("max_displacement")}
fea = pd.DataFrame({f"housing Ø{p['housing_diameter']:.0f} at {depth:g} m × {SF_p}": row(housing_res, f"{p_ext:.3f} MPa"),
                    "housing, hoop stress by hand": {"load": f"{p_ext:.3f} MPa", "max von Mises [MPa]": sigma_hoop, "SF yield": AL6082.yield_strength / sigma_hoop, "max displacement [mm]": float("nan")},
                    "housing, buckling by hand (long tube)": {"load": f"p_cr {p_cr:.2f} MPa", "max von Mises [MPa]": float("nan"), "SF yield": p_cr / p_ext, "max displacement [mm]": float("nan")},
                    "leg: pressing tripod × 2 + current": row(leg_res, f"{leg_F:.1f} N + {leg_side:.1f} N"),
                    f"tail segment 1: {Tmax:.0f} N across its end": row(tail_res_fea, f"{Tmax:.0f} N")}).T
print(f"buckling of the housing tube at {p_cr:.2f} MPa = {p_cr / (RHO * G) * 1e6:.0f} m of water (Nefri's Ø110 × 3: 353 m)")
fea.round(3)

In [ ]:
tviz.show(tviz.plot_results(housing_res, field="von_mises"))

In [ ]:
m_out = (b["parts"]["shroud + propeller (PA12, CAD)"][0] + V.thruster["motor"].mass_kg + 2 * act.get(V.servo_keys["tail"]).mass_g / 1000
         + V.parts_kg["tail segments 2x (PA12 tube, foam-filled)"] / 2)
m_added = RHO * math.pi * (p["shroud_inner"] / 2000 + p["shroud_wall"] / 1000) ** 2 * p["shroud_length"] / 1000      # the water the shroud carries
tm = talos.StructuralModel(cad["tail_segment"].artifacts["step"], "mm-N-MPa", PA12CF, TAIL_REGIONS, [talos.FixedSupport("knuckle")], [],
                           talos.MeshSettings(element_size=6.0, min_element_size=2.5), name="tail_modal", masses=[talos.PointMass("end", (m_out + m_added) * 1e-3)])
tm.mesh(RUNS / "tail_modal"); tail_modes = tm.solve_modes(RUNS / "tail_modal", n_modes=4)
f_tail = tail_modes.metrics["frequencies_hz"]
kp_tail = V.gains["tail"][0]
lever = (p["tail_segment_1"] + p["tail_segment_2"] + p["shroud_length"] / 2) / 1000
I_tail = (m_out + m_added) * lever ** 2 + V.gains["tail"][2]
f_servo = math.sqrt(kp_tail / I_tail) / (2 * math.pi)
rpm_op = np.array([1800, 2000, 2800])
orders = pd.DataFrame({"shaft [Hz]": rpm_op / 60, "blade pass [Hz]": 3 * rpm_op / 60}, index=[f"{r} rpm" for r in rpm_op])
print(f"tail segment with {m_out + m_added:.2f} kg at its end (incl. {m_added:.2f} kg of water in the shroud): {[round(f, 0) for f in f_tail]} Hz | "
      f"the tail servo's own mode (kp {kp_tail} N·m/rad on {I_tail:.4f} kg·m²): {f_servo:.1f} Hz")
fig, ax = plt.subplots(figsize=(9, 3.6))
rr = np.linspace(0, 3000, 50)
ax.plot(rr, rr / 60, label="shaft (1×)"); ax.plot(rr, 3 * rr / 60, label="blade pass (3×)")
for f in f_tail[:3]:
    ax.axhline(f, color="r", ls="--", lw=0.8)
ax.axhline(f_servo, color="k", ls=":", lw=0.8, label="tail servo mode")
ax.set(xlabel="rpm", ylabel="Hz", title="Campbell diagram: the tail segment's modes (dashed) vs the propeller's orders", ylim=(0, max(200, f_tail[0] * 1.2))); ax.legend(fontsize=8); ax.grid(alpha=0.3)
orders.round(1)

## 8. Endurance

The 4S6P pack (414 Wh, 85 % usable) against the four states of notebook 24 §8 — standby 4 W (a bigger computer),
watch 14 W (cameras, sonar), walk (the twelve leg servos at their torque share of stall current, from §2's walk),
swim (the thruster at 2000 rpm) — plus the **payload** (a sensor head or a carousel, 2 W when working: an input).
The 24 h deployment line: 20 min of walking, 5 min of swimming, the rest on standby with the sensors on 10 % of the
time and the payload working 10 % of the time.

In [ ]:
E = V.thruster["battery"].usable_wh
P_standby, P_watch, P_payload = 4.0, 14.0, 2.0                        # inputs [W]
tau_w = np.abs(np.asarray(ep_walk.log["tau"])); tw_sel = (np.asarray(ep_walk.log["t"]) > t0 + 1.3) & (np.asarray(ep_walk.log["t"]) < t1)
P_legs = sum((tau_w[tw_sel, names.index(f"{leg}_{j}")] / LEG.stall_Nm * LEG.stall_A * LEG.voltage_V).mean() for leg in lr.LEGS for j in ("yaw", "lift"))
Q_sw = lr.thrust(tab, MS["swim_rpm"], 0.8)[1]
V_sw, I_sw = mot.voltage_for(MS["swim_rpm"], Q_sw); P_thr = V_sw * I_sw
states = pd.DataFrame({"power [W]": {"standby": P_standby, "watch": P_watch, "walk": P_watch + P_legs, "swim": P_watch + P_thr, "payload working (on top)": P_payload}})
states["hours on the pack"] = E / states["power [W]"]
profile = {"walk": 20 / 60, "swim": 5 / 60}
rest = 24 - sum(profile.values())
watch_duty, payload_duty = 0.10, 0.10                                 # inputs
E24 = (profile["walk"] * states.loc["walk", "power [W]"] + profile["swim"] * states.loc["swim", "power [W]"]
       + rest * ((1 - watch_duty) * P_standby + watch_duty * P_watch + payload_duty * P_payload))
print(f"pack {V.thruster['battery'].energy_wh:.0f} Wh, {E:.0f} Wh usable (Nefri {N.thruster['battery'].usable_wh:.0f}) | a 24 h deployment uses {E24:.0f} Wh ({E24 / E * 100:.0f} % of the pack): "
      f"{24 * E / E24:.0f} h at that rate")
states.round(1)

## 9. The jobs in MuJoCo (ChironLab, underwater)

The same three missions as Nefri's (`lobster_scenario`, `Scene("ornatus")`: the cable instead of the rope, the stakes
at Ornatus's chest height, the variant's mission tuning — 2000 rpm for the swim, the press at 1800 rpm through a point
0.10 m behind the hull centre, the cutting pose 0.38 m ahead): **swim and land**; **cut the cable and enter the pipe**
— the cable parts only when both jaws squeeze it with 2.5 kN for 100 ms; **hold in a 0.5 m/s current**. The
drag of the current on the bigger shell is 1.7× Nefri's, its wet weight 2.1×: standing, it slides a little later.
Each writes a movie (`output/25_sikarian_lobster_ornatus/`; also `scenarios/sikarian_lobster.py --variant ornatus`).

In [ ]:
def mission(kind):
    scene = ls.Scene("ornatus")
    extra = dict(wind=(0.0, 0.5, 0.0), course_extent=(-2.0, 6.0, -2.0, 6.0)) if kind == "current" else {}
    lab = ls.make_lab(scene, kind, robot=robot, log_geoms=True, **extra)
    ep = ls.run(kind, scene, lab=lab)
    ep.save(RUNS / f"mission_{kind}.npz")
    return ep, ls.timeseries(ep), scene
def movie_of(ep, name, camera="follow"):
    dt = float(ep.log["t"][1] - ep.log["t"][0]); every = max(1, int(round(1 / (25 * dt))))
    imgs = cviz.frames(ep, camera=camera, every=every, size=(960, 540))
    return cviz.to_video(imgs, OUT / f"{name}.mp4", fps=25), imgs, every * dt
def starts(ep):
    out = {}
    for t0_, name, note in ep.log["mission"]:
        if note == "start": out.setdefault(name, t0_)
    return out
ep_swim, ts_swim, _ = mission("swim")
st_swim = starts(ep_swim)
sw = ts_swim[(ts_swim.t > st_swim["swim"] + 2) & (ts_swim.t < st_swim.get("ease off and sink", ts_swim.t.iloc[-1]))]
print(f"swim: finished {ep_swim.log['mission_finished']} | cruise {sw.vx.mean():.2f} m/s at {sw.z.mean():.2f} m (hull centre), pitch within ±{sw.tilt_deg.abs().max():.1f}° | "
      f"landed at x = {ts_swim.x.iloc[-1]:.2f} m, tilt {ts_swim.tilt_deg.iloc[-1]:.1f}°")
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
ax[0].plot(ts_swim.t, ts_swim.z, label="hull z [m]"); ax[0].plot(ts_swim.t, ts_swim.vx, label="vx [m/s]"); ax[0].plot(ts_swim.t, ts_swim.tilt_deg / 10, label="tilt / 10 [deg]")
ax[0].set(xlabel="time [s]", title="swim and land"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
for j in lr.TAIL_JOINTS:
    ax[1].plot(ts_swim.t, ts_swim[j + "_deg"], label=j)
ax[1].set(xlabel="time [s]", ylabel="deg", title="tail joints"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
for a in ax:
    for t_ in st_swim.values(): a.axvline(t_, color="#ddd", lw=0.6, zorder=0)
fig.tight_layout()
mv_swim, _, _ = movie_of(ep_swim, "ornatus_swim_and_land")
display(Video(str(mv_swim), embed=False, width=900))

In [ ]:
ep_cut, ts_cut, sc_cut = mission("cut_and_enter")
st_cut = starts(ep_cut)
hist = np.asarray(ep_cut.log["cutter_history"]).reshape(-1, 3)
print(f"cut and enter: finished {ep_cut.log['mission_finished']} | cable cut at {ep_cut.log['cut_at']} s | events {ep_cut.log.get('events', [])}")
print(f"Ornatus ends {ts_cut.x.iloc[-1] - sc_cut.x_mouth:.2f} m inside the pipe, tilt {ts_cut.tilt_deg.iloc[-1]:.1f}°")
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
ax[0].plot(ts_cut.t, ts_cut.x, label="hull x [m]"); ax[0].axhline(sc_cut.x_mouth, color="k", ls=":", lw=0.8, label="the pipe's mouth")
ax[0].plot(ts_cut.t, ts_cut.tilt_deg / 10, label="tilt / 10 [deg]"); ax[0].set(xlabel="time [s]", title="cut the cable and enter"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
if len(hist):
    ax[1].plot(hist[:, 0], hist[:, 1] / 1000, label="upper jaw on the cable [kN]"); ax[1].plot(hist[:, 0], hist[:, 2] / 1000, label="lower jaw [kN]")
ax[1].axhline(ls.CABLE["f_cut_N"] / 1000, color="r", ls="--", lw=0.8, label="the cutting force"); ax[1].set(xlabel="time [s]", title="the squeeze (WireCutter)"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
for a in ax:
    for t_ in st_cut.values(): a.axvline(t_, color="#ddd", lw=0.6, zorder=0)
fig.tight_layout()
mv_cut, imgs_cut, dt_cut = movie_of(ep_cut, "ornatus_cut_and_enter")
fig, axes = plt.subplots(2, 3, figsize=(18, 6.5))
for a, (name, off) in zip(axes.flat, (("walk to the rope", 5.0), ("R claw: cutting pose, jaws open", 3.5), ("cut", 0.5),
                                      ("R claw: down, the ends drop", 2.5), ("walk into the pipe", 4.0), ("walk into the pipe", 12.0))):
    j = min(len(imgs_cut) - 1, int(round((st_cut.get(name, 0.0) + off) / dt_cut)))
    a.imshow(imgs_cut[j]); a.set_axis_off(); a.set_title(f"{name} (t = {j * dt_cut:.1f} s)", fontsize=9)
fig.tight_layout()
display(Video(str(mv_cut), embed=False, width=900))

In [ ]:
ep_cur, ts_cur, _ = mission("current")
st_cur = starts(ep_cur)
def drift(a, b_):
    s = ts_cur[(ts_cur.t > a) & (ts_cur.t < b_)]
    return (s.y.iloc[-1] - s.y.iloc[0]) / (s.t.iloc[-1] - s.t.iloc[0])
v_free = drift(0.5, st_cur["curl the tail over the back"])
v_held = drift(st_cur["press down with the thrust"] + 0.5, st_cur["walk across the current, pressing"])
v_walk_c = drift(st_cur["walk across the current, pressing"] + 1.0, st_cur["stand, pressing"])
print(f"hold in a 0.5 m/s current: finished {ep_cur.log['mission_finished']} | drift standing {v_free:.3f} m/s, pressing {v_held:.3f} m/s, "
      f"walking across pressing {v_walk_c:.3f} m/s (downstream +; Nefri: 0.14, 0.02, −0.03) | max tilt {ts_cur.tilt_deg.max():.1f}°")
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(ts_cur.t, ts_cur.y, label="y (downstream) [m]"); ax.plot(ts_cur.t, ts_cur.x, label="x [m]"); ax.plot(ts_cur.t, ts_cur.foot_load_N / 10, label="feet / 10 [N]")
for t_ in st_cur.values(): ax.axvline(t_, color="#ddd", lw=0.6, zorder=0)
ax.set(xlabel="time [s]", title="in the current: slide, press, walk across"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
mv_cur, _, _ = movie_of(ep_cur, "ornatus_hold_in_current")
display(Video(str(mv_cur), embed=False, width=900))

## 10. Export, and Nefri against Ornatus

In [ ]:
summary = pd.DataFrame({"Nefri": {"mass [kg]": bn["mass"], "shell [m]": pn["shell_length"] / 1000, "wet weight [N]": bn["wet_weight_N"], "foot span [m]": pipe_fit.loc["Nefri", "foot span [m]"],
                                  "cuts": ls.ROPE["name"] + f" ({ls.ROPE['f_cut_N']:.0f} N)", "squeeze at the notch, stall [N]": act.get(N.servo_keys["jaw"]).stall_Nm / gn["cutter"],
                                  "thrust at the swim rpm [N]": lr.thrust(tab_n, N.mission["swim_rpm"], 0)[0], "walk [m/s]": 0.10, "pack [Wh]": N.thruster["battery"].energy_wh, "payload [kg]": 0.0},
                        "Ornatus": {"mass [kg]": b["mass"], "shell [m]": p["shell_length"] / 1000, "wet weight [N]": b["wet_weight_N"], "foot span [m]": pipe_fit.loc["Ornatus", "foot span [m]"],
                                    "cuts": ls.CABLE["name"] + f" ({ls.CABLE['f_cut_N']:.0f} N)", "squeeze at the notch, stall [N]": F_stall,
                                    "thrust at the swim rpm [N]": lr.thrust(tab, MS["swim_rpm"], 0)[0], "walk [m/s]": v_walk, "pack [Wh]": V.thruster["battery"].energy_wh,
                                    "payload [kg]": V.parts_kg["payload bay (sample carousel / sensor head, in the shell)"]}})
display(summary)
doc = {"source": "25_sikarian_lobster_ornatus", "design": {"file": "designs/lobster.py", "variant": "ornatus", "parameters": p}, "job": JOB.to_dict(),
       "mass_kg": b["mass"], "ballast_kg": b["ballast_kg"], "volume_L": b["volume"] * 1000, "wet_weight_N": b["wet_weight_N"], "foam_L": b["foam_volume"] * 1000, "trim": robot.trim,
       "budget": {k: {"kg": v[0], "L": v[1] * 1000} for k, v in b["parts"].items()}, "compare": compare.to_dict(), "pipe_fit": pipe_fit.to_dict(orient="index"),
       "legs": {f"{a} | {c_}": v for (a, c_), v in legs.to_dict(orient="index").items()}, "walk_m_s": v_walk,
       "bollard": bollard.to_dict(orient="index"), "swim_speed_model_m_s": v_swim, "cavitation": cav.to_dict(orient="index"),
       "tail_envelope": envelope.to_dict(orient="index"), "cfd_prop": compare_prop.to_dict(orient="index"), "cfd_jet": jet.to_dict(orient="index"),
       "cables": cables.to_dict(orient="index"), "jaw_fea": jaw_res.metrics, "fea": fea.to_dict(orient="index"), "tail_modes_hz": f_tail, "tail_servo_mode_hz": f_servo,
       "endurance": states.to_dict(orient="index"), "deployment_24h_Wh": E24, "summary": summary.to_dict(),
       "missions": {"swim": {"finished": ep_swim.log["mission_finished"], "movie": str(mv_swim)},
                    "cut_and_enter": {"finished": ep_cut.log["mission_finished"], "cut_at": ep_cut.log["cut_at"], "movie": str(mv_cut)},
                    "current": {"finished": ep_cur.log["mission_finished"], "drift_standing": v_free, "drift_pressing": v_held, "movie": str(mv_cur)}}}
(RUNS / "sikarian_lobster_ornatus.json").write_text(json.dumps(doc, indent=2, default=str))
print("written:", RUNS / "sikarian_lobster_ornatus.json")

**What the numbers say** — read them above; the ones that set the design: the ballast (§1: how close 15 kg is to
what the parts weigh — a few hundred grams means the budget is spent; add a part and the ballast goes, add another
and it is over), the pipe fit (§2: the legs are the one place where Ornatus is not a scaled Nefri), the thruster's
current on 4S (§3: the motor, not the propeller, caps the thrust), the tail servo class (§4), the cutting envelope
(§6: PVC power cable and tether cable yes, armoured cable no — that is a hydraulic cutter's job), the housing's
buckling margin (§7) and whether each job finished (§9).

**When which:** Nefri where the pipe or the gap is small and the job is a rope or a line, carried by one person,
48 h on the bottom; Ornatus for a power or signal cable, a payload (a sensor head for inspection, a sample carousel
for environmental sampling — the uses noted in notebook 24), a current that moves Nefri, and twice the pack. Both
fit the Ø600 pipe; a Ø400 pipe is Nefri's alone.

**Next steps an engineer would take:** a cutter test on the real cable (the 2.5 kN is the input the cut stands on);
the jaw screw's duty (the lead screw heats at stall: a cut every few minutes, not continuously); a hydraulic or
higher-ratio cutter module for armoured cable as an option; the payload bay's interfaces (power, data, a sample
door); the gait's speed and the thruster's power draw on a measured hull drag; and a third member — the Ø400 pipe
would need a smaller one than Nefri.